In [ ]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')   # never printed
!pip -q install kaggle
print('--- tusimple ---')
!kaggle datasets list -s tusimple --sort-by votes
print('--- culane ---')
!kaggle datasets list -s culane --sort-by votes

In [ ]:
import os, collections
from google.colab import drive, userdata
drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')   # never printed
!pip -q install kaggle
DEST = '/content/tusimple_pre'
!kaggle datasets download -d rangalamahesh/preprocessed-1 -p {DEST} --unzip

tot, cnt, exts, samples = 0, collections.Counter(), collections.Counter(), {}
for root, _, fs in os.walk(DEST):
    for f in fs:
        tot += os.path.getsize(os.path.join(root, f))
        rel = os.path.relpath(root, DEST)
        cnt[rel] += 1
        exts[os.path.splitext(f)[1].lower()] += 1
        samples.setdefault(rel, []).append(f)
print('total GB on disk:', round(tot / 1e9, 2))
print('file types:', dict(exts))
for k in sorted(cnt):
    print(f'{cnt[k]:>6}  {k}   e.g. {sorted(samples[k])[:3]}')

In [ ]:
import os, glob, collections
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files

R = '/content/tusimple_pre/tusimple_preprocessed/training'
assert os.path.isdir(R), 'dataset folder missing - rerun the download cell first'
fr = sorted(glob.glob(f'{R}/frames/*.jpg')); mk = sorted(glob.glob(f'{R}/lane-masks/*.jpg'))
nf = [os.path.basename(p) for p in fr]
assert nf == [os.path.basename(p) for p in mk], 'frame and mask names differ'
print('pairs:', len(fr))
print('frames per drive:', dict(collections.Counter(n.split('_')[0] for n in nf)))
im, m = Image.open(fr[0]), Image.open(mk[0])
print('frame size/mode:', im.size, im.mode, '| mask size/mode:', m.size, m.mode)

rng = np.random.default_rng(0)
idx = rng.choice(len(mk), 40, replace=False)
vals = np.concatenate([np.asarray(Image.open(mk[i]).convert('L')).ravel() for i in idx])
u, cnt = np.unique(vals, return_counts=True)
order = np.argsort(-cnt)[:6]
print('distinct mask values:', len(u), '| most common (value, share %):',
      [(int(u[i]), round(100 * cnt[i] / vals.size, 2)) for i in order])
print('share of pixels > 127:', round(100 * float((vals > 127).mean()), 2), '%')

fig, ax = plt.subplots(3, 2, figsize=(10, 9))
for r, i in enumerate(rng.choice(len(fr), 3, replace=False)):
    ax[r, 0].imshow(Image.open(fr[i])); ax[r, 0].set_title(nf[i]); ax[r, 0].axis('off')
    ax[r, 1].imshow(Image.open(mk[i]).convert('L'), cmap='gray'); ax[r, 1].set_title('lane mask'); ax[r, 1].axis('off')
plt.tight_layout(); plt.savefig('/content/tusimple_samples.png', dpi=110); plt.show()
files.download('/content/tusimple_samples.png')

In [ ]:
import os, sys, glob, time, json, subprocess
import numpy as np, cv2, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from google.colab import drive, userdata, files

drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
dev = 'cuda'
BASE = '/content/drive/MyDrive/adp_project'
R = '/content/tusimple_pre/tusimple_preprocessed/training'
if not os.path.isdir(R):
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')   # never printed
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'kaggle'], check=True)
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'rangalamahesh/preprocessed-1',
                    '-p', '/content/tusimple_pre', '--unzip'], check=True)
assert os.path.isdir(R), 'dataset folder missing'
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'segmentation-models-pytorch', 'wandb'], check=True)
import segmentation_models_pytorch as smp

# ---- split by drive ----
names = sorted(os.path.basename(p) for p in glob.glob(f'{R}/frames/*.jpg'))
drv = lambda n: n.split('_')[0]
SPLIT = {'train': [n for n in names if drv(n) in ('0313-1', '0313-2')],
         'val':   [n for n in names if drv(n) == '0531'],
         'test':  [n for n in names if drv(n) == '0601']}
print({k: len(v) for k, v in SPLIT.items()})
assert all(len(v) > 0 for v in SPLIT.values()), 'a split is empty'
os.makedirs(f'{BASE}/logs', exist_ok=True)
json.dump(SPLIT, open(f'{BASE}/logs/tusimple_split.json', 'w'))

W, H = 640, 352
def load(ns):
    X = np.zeros((len(ns), H, W, 3), np.uint8); Y = np.zeros((len(ns), H, W), np.uint8)
    for i, n in enumerate(ns):
        im = cv2.imread(f'{R}/frames/{n}')
        X[i] = cv2.cvtColor(cv2.resize(im, (W, H), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
        mk = cv2.imread(f'{R}/lane-masks/{n}', cv2.IMREAD_GRAYSCALE)
        Y[i] = (cv2.resize(mk, (W, H), interpolation=cv2.INTER_AREA) > 100).astype(np.uint8)
    return X, Y
t0 = time.time()
D = {k: load(v) for k, v in SPLIT.items()}
print(f'data loaded in {time.time()-t0:.0f}s | lane pixel share (train): {100*D["train"][1].mean():.2f}%')

mean = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
def batch(Xs, Ys, idx, aug):
    x = torch.from_numpy(Xs[idx]).to(dev).permute(0, 3, 1, 2).float() / 255.
    y = torch.from_numpy(Ys[idx]).to(dev).float().unsqueeze(1)
    if aug:
        x = (x * torch.empty(len(idx), 1, 1, 1, device=dev).uniform_(0.7, 1.3)).clamp(0, 1)
        fl = torch.rand(len(idx), device=dev) < 0.5
        x[fl] = x[fl].flip(-1); y[fl] = y[fl].flip(-1)
    return (x - mean) / std, y

try:
    model = smp.Unet('resnet18', encoder_weights='imagenet', in_channels=3, classes=1); INIT = 'imagenet'
except Exception as e:
    print('ImageNet weights failed:', type(e).__name__, str(e)[:150], '-> training from scratch')
    model = smp.Unet('resnet18', encoder_weights=None, in_channels=3, classes=1); INIT = 'scratch'
model = model.to(dev)
print('encoder init:', INIT, '| params (M):', round(sum(p.numel() for p in model.parameters()) / 1e6, 1))

def loss_fn(lg, y):
    p = torch.sigmoid(lg)
    inter = (p * y).sum((1, 2, 3))
    dice = 1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)
    return F.binary_cross_entropy_with_logits(lg, y) + dice.mean()

@torch.no_grad()
def evaluate(Xs, Ys, bs=16):
    model.eval(); tp = fp = fn = 0
    for i in range(0, len(Xs), bs):
        x, y = batch(Xs, Ys, np.arange(i, min(i + bs, len(Xs))), False)
        with torch.autocast('cuda', dtype=torch.float16):
            p = torch.sigmoid(model(x)) > 0.5
        yb = y > 0.5
        tp += (p & yb).sum().item(); fp += (p & ~yb).sum().item(); fn += (~p & yb).sum().item()
    return tp / (tp + fp + fn + 1e-9), 2 * tp / (2 * tp + fp + fn + 1e-9)

# ---- W&B (optional) ----
use_wb = False
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')         # never printed
    import wandb; wandb.login()
    wandb.init(project='adp-lanes', name='unet-resnet18-tusimple', id='unet-r18-tusimple-v1', resume='allow')
    use_wb = True
except Exception as e:
    print('W&B off:', type(e).__name__)

# ---- training (resumable) ----
EPOCHS, BS = 15, 16
CK = f'{BASE}/models/lane_unet'; os.makedirs(CK, exist_ok=True)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')
start, best, hist = 0, -1.0, []
if os.path.exists(f'{CK}/last.pt'):
    ck = torch.load(f'{CK}/last.pt', map_location=dev)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
    start, best, hist = ck['epoch'], ck['best'], ck['hist']
    print('resumed from epoch', start)
Xtr, Ytr = D['train']
for ep in range(start, EPOCHS):
    model.train(); t1 = time.time(); order = np.random.permutation(len(Xtr)); tl, nb = 0.0, 0
    for i in range(0, len(order) - BS + 1, BS):
        x, y = batch(Xtr, Ytr, order[i:i + BS], True)
        with torch.autocast('cuda', dtype=torch.float16):
            lg = model(x)
        loss = loss_fn(lg.float(), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tl += loss.item(); nb += 1
    sched.step()
    viou, vdice = evaluate(*D['val'])
    hist.append(dict(epoch=ep + 1, train_loss=round(tl / nb, 4), val_iou=round(viou, 4), val_dice=round(vdice, 4)))
    print(hist[-1], f'{time.time()-t1:.0f}s')
    if use_wb: wandb.log(dict(hist[-1]))
    if viou > best:
        best = viou; torch.save(model.state_dict(), f'{CK}/best.pt')
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                    epoch=ep + 1, best=best, hist=hist), f'{CK}/last.pt')
json.dump(hist, open(f'{BASE}/logs/lane_unet_history.json', 'w'), indent=1)

# ---- final test evaluation with the best (by validation) checkpoint ----
model.load_state_dict(torch.load(f'{CK}/best.pt', map_location=dev))
tiou, tdice = evaluate(*D['test'])
def tolerant(Xs, Ys, tol=4):
    k = np.ones((2 * tol + 1, 2 * tol + 1), np.uint8); pp = pm = gp = gm = 0
    model.eval()
    with torch.no_grad():
        for i in range(0, len(Xs), 16):
            idx = np.arange(i, min(i + 16, len(Xs))); x, _ = batch(Xs, Ys, idx, False)
            with torch.autocast('cuda', dtype=torch.float16):
                pred = (torch.sigmoid(model(x)) > 0.5).squeeze(1).cpu().numpy().astype(np.uint8)
            for p_, g_ in zip(pred, Ys[idx]):
                pd_, gd_ = cv2.dilate(p_, k), cv2.dilate(g_, k)
                pp += p_.sum(); pm += (p_ & gd_).sum(); gp += g_.sum(); gm += (g_ & pd_).sum()
    pr, rc = pm / max(pp, 1), gm / max(gp, 1)
    return pr, rc, 2 * pr * rc / max(pr + rc, 1e-9)
pr, rc, f1 = tolerant(*D['test'])
x1, _ = batch(*D['test'], np.arange(1), False)
with torch.no_grad():
    for _ in range(10): model(x1)
    torch.cuda.synchronize(); ts = time.perf_counter()
    for _ in range(50): model(x1)
    torch.cuda.synchronize(); ms = (time.perf_counter() - ts) / 50 * 1000
res = dict(test_images=len(SPLIT['test']), best_val_iou=round(best, 4), test_iou=round(tiou, 4), test_dice=round(tdice, 4),
           tol4px_precision=round(pr, 4), tol4px_recall=round(rc, 4), tol4px_f1=round(f1, 4),
           model_ms_per_image_t4_fp32_batch1=round(ms, 1), input=f'{W}x{H}', encoder_init=INIT,
           note='tolerance 4 px at 640x352 (about 8 px at 1280x720); split by drive; one highway dataset')
print(json.dumps(res, indent=1))
if use_wb: wandb.log({'test_' + k: v for k, v in res.items() if isinstance(v, (int, float))}); wandb.finish()
json.dump(res, open(f'{BASE}/logs/lane_unet_test.json', 'w'), indent=1)

# ---- picture: 4 test frames (image | ground truth in green | prediction in red) ----
Xt, Yt = D['test']; sel = np.linspace(0, len(Xt) - 1, 4).astype(int)
x, _ = batch(Xt, Yt, sel, False)
with torch.no_grad(): pr_ = (torch.sigmoid(model(x)) > 0.5).squeeze(1).cpu().numpy()
fig, ax = plt.subplots(4, 3, figsize=(12, 9))
for r, i in enumerate(sel):
    ov = Xt[i].copy(); ov[pr_[r] > 0] = (255, 0, 0)
    gv = Xt[i].copy(); gv[Yt[i] > 0] = (0, 255, 0)
    for c, (im, t) in enumerate(((Xt[i], SPLIT['test'][i]), (gv, 'ground truth'), (ov, 'prediction'))):
        ax[r, c].imshow(im); ax[r, c].set_title(t, fontsize=8); ax[r, c].axis('off')
plt.tight_layout(); plt.savefig('/content/lane_seg_examples.png', dpi=110); plt.show()
for f in ('/content/lane_seg_examples.png', f'{BASE}/logs/lane_unet_test.json',
          f'{BASE}/logs/lane_unet_history.json', f'{BASE}/logs/tusimple_split.json'):
    files.download(f)

In [ ]:
import os, sys, json, time, types, subprocess
import numpy as np, cv2, torch
import matplotlib.pyplot as plt
from google.colab import files

assert 'D' in globals() and 'model' in globals() and 'batch' in globals(), \
    'Variables are gone (runtime reset?). Rerun the training cell first: it resumes from the saved checkpoint, skips training and redoes the evaluation.'
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
if not os.path.isdir('/content/repo'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/DarshanKumarGP/autonomous-driving-perception', '/content/repo'], check=True)
sys.path.insert(0, '/content/repo/src')
import lane_classical as lc

Xv, Yv = D['val']; Xt, Yt = D['test']
H_, W_ = Yv.shape[1:]
K = np.ones((9, 9), np.uint8)                      # 4 px tolerance on each side

def make_args(cfg):
    return types.SimpleNamespace(top=cfg[0], bottom=0.98, top_half_width=cfg[1], bottom_half_width=0.5,
                                 hough_threshold=cfg[2], alpha=0.25, hold=0, no_color=False)

def classical_masks(X, cfg):
    a = make_args(cfg)
    out = np.zeros(X.shape[:3], np.uint8); both = 0
    for i in range(len(X)):
        fr = cv2.cvtColor(X[i], cv2.COLOR_RGB2BGR)
        st = {}
        raw = lc.detect(fr, a, st)                 # fresh state per frame: no temporal smoothing
        both += int(raw['left'] is not None and raw['right'] is not None)
        canvas = lc.draw(np.zeros_like(fr), a, st)  # draw on black, keep the pure red line pixels
        out[i] = ((canvas[..., 2] > 200) & (canvas[..., 1] < 60) & (canvas[..., 0] < 60)).astype(np.uint8)
    return out, both

def roi_mask(cfg):
    m = np.zeros((H_, W_), np.uint8)
    cv2.fillPoly(m, lc.roi_polygon(W_, H_, make_args(cfg)), 1)
    return m

def score(P, G, roi=None):
    pp = pm = gp = gm = 0
    for p, g in zip(P, G):
        if roi is not None:
            p, g = p & roi, g & roi
        pd_, gd_ = cv2.dilate(p, K), cv2.dilate(g, K)
        pp += int(p.sum()); pm += int((p & gd_).sum()); gp += int(g.sum()); gm += int((g & pd_).sum())
    pr, rc = pm / max(pp, 1), gm / max(gp, 1)
    return dict(precision=round(pr, 3), recall=round(rc, 3), f1=round(2 * pr * rc / max(pr + rc, 1e-9), 3))

# 1) choose the classical settings on the VALIDATION drive (full-frame F1)
CFGS = [(0.45, 0.08, 30), (0.50, 0.10, 30), (0.45, 0.12, 20), (0.55, 0.08, 20)]   # (top, top_half_width, hough_threshold)
best_cfg, best_f1 = None, -1
for cfg in CFGS:
    P, both = classical_masks(Xv, cfg)
    s = score(P, Yv)
    print('val', cfg, s, '| both lines found in', round(100 * both / len(Xv)), '% of frames')
    if s['f1'] > best_f1:
        best_f1, best_cfg = s['f1'], cfg
print('chosen classical config:', best_cfg)

# 2) classical on the TEST drive
t0 = time.perf_counter()
Pc, both_c = classical_masks(Xt, best_cfg)
ms_c = (time.perf_counter() - t0) / len(Xt) * 1000
roi = roi_mask(best_cfg)

# 3) segmentation model on the TEST drive (best checkpoint chosen on validation)
model.load_state_dict(torch.load(f'{CK}/best.pt', map_location=dev)); model.eval()
Ps = []
with torch.no_grad():
    for i in range(0, len(Xt), 16):
        idx = np.arange(i, min(i + 16, len(Xt))); x, _ = batch(Xt, Yt, idx, False)
        with torch.autocast('cuda', dtype=torch.float16):
            Ps.append((torch.sigmoid(model(x)) > 0.5).squeeze(1).cpu().numpy().astype(np.uint8))
Ps = np.concatenate(Ps)

res = dict(test_frames=len(Xt),
           classical_config=dict(top=best_cfg[0], top_half_width=best_cfg[1], hough_threshold=best_cfg[2], bottom=0.98, bottom_half_width=0.5),
           classical_chosen_on='validation drive 0531 by full-frame F1',
           full_frame=dict(classical=score(Pc, Yt), segmentation=score(Ps, Yt)),
           inside_classical_region=dict(classical=score(Pc, Yt, roi), segmentation=score(Ps, Yt, roi)),
           classical_both_lines_found_pct=round(100 * both_c / len(Xt), 1),
           classical_ms_per_frame_colab_cpu=round(ms_c, 1), segmentation_ms_per_frame_t4=15.8,
           note='4 px tolerance at 640x352; one highway dataset; classical outputs only the two lines around the ego lane')
print(json.dumps(res, indent=1))
json.dump(res, open(f'{BASE}/logs/lane_comparison.json', 'w'), indent=1)

sel = np.linspace(0, len(Xt) - 1, 4).astype(int)
fig, ax = plt.subplots(4, 3, figsize=(12, 9))
for r, i in enumerate(sel):
    g = Xt[i].copy(); g[Yt[i] > 0] = (0, 255, 0)
    c = Xt[i].copy(); c[Pc[i] > 0] = (0, 120, 255)
    s_ = Xt[i].copy(); s_[Ps[i] > 0] = (255, 0, 0)
    for k, (im, t) in enumerate(((g, 'ground truth'), (c, 'classical OpenCV'), (s_, 'U-Net segmentation'))):
        ax[r, k].imshow(im); ax[r, k].set_title(t, fontsize=8); ax[r, k].axis('off')
plt.tight_layout(); plt.savefig('/content/lane_comparison.png', dpi=110); plt.show()
files.download('/content/lane_comparison.png'); files.download(f'{BASE}/logs/lane_comparison.json')